# BIOSTAT 707 Checkpoint 1

Author: Ruiyang Hu

## Setup



In [ ]:
import os
import sys
from pathlib import Path

# Configure Windows DLL lookup before importing scientific packages.
# Keep this handle alive for the lifetime of the kernel.
if sys.platform == 'win32':
    dll_dir = Path(sys.prefix) / 'Library' / 'bin'
    if dll_dir.is_dir():
        dll_directory_handle = os.add_dll_directory(str(dll_dir))
        if str(dll_dir) not in os.environ.get('PATH', '').split(os.pathsep):
            os.environ['PATH'] = str(dll_dir) + os.pathsep + os.environ.get('PATH', '')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('.')
DATA = ROOT / 'data'
SET_A = DATA / 'set-a'
OUTPUT = ROOT / 'output'
OUTPUT.mkdir(exist_ok=True)

## Data loading

Find the Set A patient text files in a stable order. Should have 4000.

In [ ]:
patient_files = sorted(SET_A.glob('*.txt'))
if len(patient_files) != 4000:
    raise ValueError(f'Expected 4000 patient files in {SET_A}, found {len(patient_files)}.')
print(f'Found {len(patient_files)} Set A patient files.')

Inspect one raw file before combining the patients. Its rows are a skinny table of time, parameter, and value; the `RecordID` row is patient metadata.

In [ ]:
example_file = patient_files[0]
example = pd.read_csv(example_file)
print(f'Example file: {example_file}')
display(example.head())
print('Shape:', example.shape)
print('Columns:', example.columns.tolist())
print('Unique Parameters:', example['Parameter'].unique())
print('Parameter value counts:')
display(example['Parameter'].value_counts())

## Build long table

Read each patient file and require exactly one internal `RecordID` that matches its filename. Remove that metadata row, add the verified ID to each clinical observation, and convert elapsed `HH:MM` to numeric hours while keeping the original `Time`. Keep raw `Value` entries, including `-1`, for later variable-specific missing-value decisions.

In [ ]:
patient_tables = []
for patient_file in patient_files:
    patient = pd.read_csv(patient_file, dtype={'Value': str})
    record_ids = patient.loc[patient['Parameter'] == 'RecordID', 'Value']
    if len(record_ids) != 1 or str(record_ids.iloc[0]) != patient_file.stem:
        raise ValueError(f'RecordID row does not match filename in {patient_file}.')

    clinical = patient.loc[patient['Parameter'] != 'RecordID'].copy()
    clinical['Value'] = pd.to_numeric(clinical['Value'], errors='raise')
    clinical.insert(0, 'RecordID', patient_file.stem)
    patient_tables.append(clinical)

set_a_long = pd.concat(patient_tables, ignore_index=True)
if not set_a_long['Time'].str.fullmatch(r'\d{2}:\d{2}').all():
    raise ValueError('Some Time values are not in HH:MM format.')
hours_minutes = set_a_long['Time'].str.split(':', expand=True).astype(int)
if not hours_minutes[1].between(0, 59).all():
    raise ValueError('Some Time values have minutes outside 00–59.')
set_a_long['hours_since_admission'] = hours_minutes[0] + hours_minutes[1] / 60
set_a_long.to_csv(OUTPUT / 'set-a_long.csv', index=False)

## Data sanity checks

Check patient coverage, table size, elapsed time, duplicate observations, missing fields, and parameter coverage. Show rows from both ends of the combined table. Duplicate rows are reported for inspection; repeated source observations are kept.

In [ ]:
print('Unique RecordIDs:', set_a_long['RecordID'].nunique())
print('Shape:', set_a_long.shape)
print('Hours since admission (min, max):',
      set_a_long['hours_since_admission'].min(),
      set_a_long['hours_since_admission'].max())
print('Duplicated rows:', set_a_long.duplicated().sum())
print('Missing values in required fields:')
display(set_a_long[['RecordID', 'Time', 'Parameter', 'Value']].isna().sum())
print('Unique Parameters:', set_a_long['Parameter'].nunique())
print('First rows:')
display(set_a_long.head())
print('Last rows:')
display(set_a_long.tail())

The combined long table contained 1,753,980 observations from all 4,000 ICU stays. After using the `RecordID` metadata field as the patient identifier and removing it from the longitudinal measurements, 41 unique clinical parameters remained. Recorded times ranged from 0 to 48 hours after ICU admission, with no missing identifiers or exact duplicate rows.

Inspect raw distributions before cleaning. Counts below refer to observations, not patients. I preserve `Value` and replace only the measurement sentinel `-1` with `NaN` in `Value_clean`; the saved long CSV retains its original five-column schema. Quantiles describe the data and are not exclusion thresholds.

In [ ]:
raw_columns = set_a_long.columns.tolist()
assert set_a_long.shape == (1753980, 5)
assert set_a_long['RecordID'].nunique() == 4000
assert set_a_long['hours_since_admission'].between(0, 48).all()
raw_group = set_a_long.groupby('Parameter')['Value']
raw_summary = raw_group.agg(count='count', minimum='min', median='median', mean='mean', maximum='max')
raw_summary['n_minus1'] = set_a_long['Value'].eq(-1).groupby(set_a_long['Parameter']).sum()
raw_summary['pct_minus1'] = 100 * raw_summary['n_minus1'] / raw_summary['count']
raw_summary['p01'] = raw_group.quantile(0.01)
raw_summary['p99'] = raw_group.quantile(0.99)
raw_summary = raw_summary[['count', 'n_minus1', 'pct_minus1', 'minimum', 'p01', 'median', 'mean', 'p99', 'maximum']]
display(raw_summary.round(3))
raw_summary.to_csv(OUTPUT / 'raw_parameter_summary.csv')
negative = set_a_long.loc[set_a_long['Value'] < 0]
print('Total negative observations:', len(negative))
display(negative.groupby('Parameter')['Value'].agg(count='size', unique_values=lambda x: sorted(x.unique())))
set_a_long['Value_clean'] = set_a_long['Value'].mask(set_a_long['Value'].eq(-1))
print('Encoded missing measurements:', set_a_long['Value'].eq(-1).sum())
clean_summary = set_a_long.groupby('Parameter')['Value_clean'].describe(percentiles=[0.01, 0.5, 0.99])
display(clean_summary.round(3))

In [ ]:
descriptor_names = ['Age', 'Gender', 'Height', 'ICUType', 'Weight', 'MechVent']
descriptors = set_a_long.loc[set_a_long['Parameter'].isin(descriptor_names)]
display(descriptors.groupby('Parameter')['Value_clean'].agg(count='count', unique_count='nunique', minimum='min', maximum='max'))
allowed_codes = {'Gender': [0, 1], 'ICUType': [1, 2, 3, 4], 'MechVent': [0, 1]}
for parameter, allowed in allowed_codes.items():
    values = descriptors.loc[descriptors['Parameter'].eq(parameter), 'Value_clean'].dropna()
    print(parameter, 'observed codes:', sorted(values.unique()), 'unexpected:', sorted(values.loc[~values.isin(allowed)].unique()))
# Show identifiable min/max rows for selected variables; no observations are removed.
inspect_parameters = ['Height', 'Weight', 'Temp', 'pH', 'HR', 'K', 'Glucose', 'Urine']
extreme_source = set_a_long.loc[set_a_long['Parameter'].isin(inspect_parameters) & set_a_long['Value_clean'].notna()]
extreme_group = extreme_source.groupby('Parameter')['Value_clean']
extreme_indices = pd.concat([extreme_group.idxmin(), extreme_group.idxmax()]).unique()
extreme_values = set_a_long.loc[extreme_indices].sort_values(['Parameter', 'Value_clean'])
display(extreme_values)
extreme_values.to_csv(OUTPUT / 'extreme_values.csv', index=False)

The raw data have 2,223 `-1` entries (Gender, Height, and Weight). Ordinary `isna()` checks missed them because they are numeric codes, not actual missing values. Temperature also has 24 negative entries, with values -17.8 and -12.5; these are not the `-1` sentinel. Height reaches 431.8 cm and pH reaches 735, suggesting possible recording or unit problems. They remain visible in this initial analysis, so summaries involving these variables need caution and later clinical/source review. Critically ill patients can also have legitimate extreme measurements; I do not apply automatic outlier deletion or guess corrections.

Gender, ICUType, and MechVent are checked against their documented encodings. MechVent contains only recorded 1s: absence is not proof of no ventilation. Age, Height, and Weight ranges are inspected without inventing clinical exclusion thresholds.

## Temporal missingness

Measurement presence means at least one recorded, non-missing measurement, not its numeric value or clinical validity. Each patient contributes at most once per parameter-hour. Only measurement codes of -1 and actual missing values are excluded; suspicious extremes remain available for review.

After confirming the validated long CSV exists, release the earlier large objects. Read one raw patient file at a time and accumulate a small count table. Hours 0-47 represent full hours; hour 48 contains only the exact endpoint. Static descriptors (Age, Gender, Height, ICUType) are excluded from the main heatmap because they are usually recorded at admission. MechVent remains time-varying. Patterns may reflect clinical workflow and severity, so missingness may be informative rather than completely random. No causal claims are made.


In [ ]:
import gc

assert (OUTPUT / 'set-a_long.csv').is_file(), 'Run the earlier validated long-table section first.'
# GroupBy objects and the list of patient tables can retain large data too.
for object_name in [
    'set_a_long', 'patient_tables', 'patient', 'clinical', 'hours_minutes',
    'raw_group', 'negative', 'descriptors', 'values', 'extreme_source',
    'extreme_group', 'extreme_values', 'measured', 'ordered', 'static_rows',
    'repeated_rows', 'saved_long', 'temporal_data', 'presence',
]:
    globals().pop(object_name, None)
gc.collect()
plt.close('all')

static_parameters = ['Age', 'Gender', 'Height', 'ICUType']
patient_files = sorted((DATA / 'set-a').glob('*.txt'))
assert len(patient_files) == 4000

# Shared by the two patient-by-patient passes; no full long table is loaded.
def read_patient_measurements(patient_path):
    patient = pd.read_csv(
        patient_path, usecols=['Time', 'Parameter', 'Value'],
        dtype={'Time': 'string', 'Parameter': 'category', 'Value': 'float64'},
    )
    identifier = patient.loc[patient['Parameter'].eq('RecordID'), 'Value']
    assert len(identifier) == 1 and identifier.notna().all()
    record_id = int(identifier.iloc[0])
    assert identifier.iloc[0] == record_id and str(record_id) == patient_path.stem
    patient = patient.loc[patient['Parameter'].ne('RecordID')].copy()
    assert patient['Time'].str.fullmatch(r'\d{2}:\d{2}').all()
    time_parts = patient['Time'].str.split(':', expand=True).astype('int16')
    assert time_parts[1].between(0, 59).all()
    patient['elapsed_minutes'] = time_parts[0] * 60 + time_parts[1]
    assert patient['elapsed_minutes'].between(0, 48 * 60).all()
    patient['Value_clean'] = patient['Value'].mask(patient['Value'].eq(-1))
    return record_id, patient

# Verify the small denominator file without loading other outcome columns.
cohort_ids = pd.read_csv(DATA / 'Outcomes-a.txt', usecols=['RecordID'], dtype={'RecordID': 'int32'})
assert len(cohort_ids) == 4000 and cohort_ids['RecordID'].is_unique
expected_ids = set(cohort_ids['RecordID'])
n_patients = len(expected_ids)
del cohort_ids
hour_counts = {}
seen_ids = set()
for patient_path in patient_files:
    record_id, patient = read_patient_measurements(patient_path)
    assert record_id not in seen_ids
    seen_ids.add(record_id)
    for parameter in patient['Parameter'].unique():
        if parameter not in hour_counts:
            hour_counts[parameter] = np.zeros(49, dtype='int32')
    presence = patient.loc[patient['Value_clean'].notna(), ['Parameter', 'elapsed_minutes']].copy()
    presence['hour_bin'] = (presence['elapsed_minutes'] // 60).astype('int8')
    presence = presence[['Parameter', 'hour_bin']].drop_duplicates()
    for parameter, hour in presence.itertuples(index=False, name=None):
        hour_counts[parameter][hour] += 1
    del patient, presence
assert seen_ids == expected_ids
parameters = sorted(hour_counts)
assert len(parameters) == 41
hourly_counts = pd.DataFrame.from_dict(hour_counts, orient='index', columns=range(49)).sort_index()
hourly_counts.index.name = 'Parameter'
hourly_counts.columns.name = 'hour_bin'
hourly_proportion = hourly_counts / n_patients
assert hourly_proportion.shape == (41, 49)
assert hourly_proportion.ge(0).all().all() and hourly_proportion.le(1).all().all()
hourly_proportion.to_csv(OUTPUT / 'hourly_measurement_proportion.csv')
print('Patients:', n_patients, 'Shape:', hourly_proportion.shape)
print('Proportion range:', hourly_proportion.to_numpy().min(), hourly_proportion.to_numpy().max())
del hour_counts, seen_ids
gc.collect()

In [ ]:
# Run after the temporal measurement-presence calculation cell.
time_parameters = [
    parameter for parameter in parameters
    if parameter not in static_parameters
]

fig, ax = plt.subplots(figsize=(14, 10))
image = ax.imshow(
    hourly_proportion.loc[time_parameters].to_numpy(),
    aspect='auto',
    interpolation='nearest',
    cmap='viridis',
    vmin=0,
    vmax=1,
)
ax.set_xticks(range(0, 49, 4))
ax.set_yticks(range(len(time_parameters)), labels=time_parameters)
ax.set(
    xlabel='Hour since ICU admission (48 = endpoint only)',
    ylabel='Time-varying parameter',
    title='Hourly measurement presence',
)
fig.colorbar(image, ax=ax, label='Proportion of 4,000 ICU stays')
fig.tight_layout()

figure_path = OUTPUT / 'temporal_missingness.png'
fig.savefig(figure_path, dpi=110)
plt.close(fig)
del fig, ax, image
print('Figure saved to:', figure_path)

# Display static descriptors separately from the time-varying heatmap.
display(hourly_proportion.loc[static_parameters].round(3))


## Outcome data

Load the small outcome table and inspect its actual column names. Resolve the expected fields by normalized names, failing explicitly if a field is absent or ambiguous. Preserve raw outcome values: Survival = -1 means no recorded death, not a negative duration and not an ordinary measurement-missingness code. Recorded death may occur after hospital discharge. 

In [ ]:
outcomes = pd.read_csv(DATA / 'Outcomes-a.txt', dtype={'RecordID': 'int32'})
print('Outcome columns:', outcomes.columns.tolist())
assert len(outcomes) == 4000
assert outcomes['RecordID'].nunique() == 4000
assert not outcomes['RecordID'].duplicated().any()
assert set(outcomes['RecordID']) == expected_ids

def outcome_column(normalized_name):
    matches = [column for column in outcomes.columns
               if ''.join(character.lower() for character in column if character.isalnum()) == normalized_name]
    if len(matches) != 1:
        raise ValueError(f'Expected one outcome column for {normalized_name}; found {matches}')
    return matches[0]

death_column = outcome_column('inhospitaldeath')
los_column = outcome_column('lengthofstay')
saps_column = outcome_column('sapsi')
sofa_column = outcome_column('sofa')
survival_column = outcome_column('survival')
assert outcomes[death_column].isin([0, 1]).all()
display(outcomes.head())
display(outcomes.drop(columns='RecordID').describe())
display(outcomes.drop(columns='RecordID').eq(-1).sum().rename('Number coded -1'))

## Build wide table

Make a second pass through the patient files, retaining only one summary row per stay. Use one inclusive 0-48-hour window. First and last refer to chronological non-missing values; source row order breaks timestamp ties. Static descriptors receive a single first non-missing value plus a presence count. Report conflicting static values for review instead of silently treating them as repeated physiologic variables.

Weight receives longitudinal summaries because it can vary. A separate admission-time weight (time zero only) is used in Table 1. Missing admission weight is not replaced with a later weight. Measurement -1 codes do not contribute to summaries, and other suspicious values remain unchanged. This initial representation supports cohort characterization and may be revised later; no model is built.

In [ ]:
wide_rows = []
static_conflicts = []
for patient_path in patient_files:
    record_id, patient = read_patient_measurements(patient_path)
    patient = patient.sort_values('elapsed_minutes', kind='stable')
    row = {'RecordID': record_id}
    # Initialize absent parameters explicitly, including static presence counts.
    for parameter in parameters:
        row[f'{parameter}_count'] = 0
        if parameter in static_parameters:
            row[parameter] = np.nan
        else:
            for statistic in ['first', 'last', 'min', 'max', 'mean']:
                row[f'{parameter}_{statistic}'] = np.nan
    for parameter, group in patient.groupby('Parameter', observed=True, sort=False):
        valid = group['Value_clean'].dropna()
        row[f'{parameter}_count'] = len(valid)
        if valid.empty:
            continue
        if parameter in static_parameters:
            row[parameter] = valid.iloc[0]
            if valid.nunique() > 1:
                static_conflicts.append({'RecordID': record_id, 'Parameter': parameter,
                                         'values': valid.unique().tolist()})
        else:
            row[f'{parameter}_first'] = valid.iloc[0]
            row[f'{parameter}_last'] = valid.iloc[-1]
            row[f'{parameter}_min'] = valid.min()
            row[f'{parameter}_max'] = valid.max()
            row[f'{parameter}_mean'] = valid.mean()
    admission_weight = patient.loc[patient['Parameter'].eq('Weight') & patient['elapsed_minutes'].eq(0), 'Value_clean'].dropna()
    row['Weight_admission'] = admission_weight.iloc[0] if len(admission_weight) else np.nan
    wide_rows.append(row)
    del patient, group, valid, admission_weight, row

set_a_wide = pd.DataFrame(wide_rows)
del wide_rows
gc.collect()
count_columns = [f'{parameter}_count' for parameter in parameters]
set_a_wide[count_columns] = set_a_wide[count_columns].astype('int32')
assert len(set_a_wide) == 4000 and set_a_wide['RecordID'].is_unique
assert set(set_a_wide['RecordID']) == set(outcomes['RecordID'])
set_a_wide = set_a_wide.merge(outcomes, on='RecordID', how='left', validate='one_to_one', indicator=True)
assert set_a_wide['_merge'].eq('both').all()
set_a_wide.drop(columns='_merge', inplace=True)
assert len(set_a_wide) == 4000 and not set_a_wide['RecordID'].duplicated().any()
set_a_wide.to_csv(OUTPUT / 'set-a_wide.csv', index=False)
print('Wide shape:', set_a_wide.shape)
print('Conflicting static patient/parameter pairs:', len(static_conflicts))
display(pd.DataFrame(static_conflicts).head(10))
display(set_a_wide.head())

## Table 1

Describe the whole cohort and groups defined by in-hospital death. Continuous variables use median [Q1, Q3] among available values, with missing counts shown separately. Categorical entries are n (%) using all patients in each group as the denominator, including a missing category. Preserve unexpected categorical encodings as labeled values for review.

Use admission-time Weight. SAPS-I and SOFA are supplied severity scores and are not necessarily admission-time measurements. Negative score codes are excluded only in this descriptive copy, since they are not interpretable scores; the original outcomes remain unchanged. No tests or adjusted comparisons are performed. Inspect group differences and missing counts when interpreting the table.

In [ ]:
table_data = set_a_wide[['Age', 'Gender', 'ICUType', 'Height', 'Weight_admission',
                             saps_column, sofa_column, death_column]].copy()
for column in [saps_column, sofa_column]:
    table_data[column] = table_data[column].mask(table_data[column].lt(0))
category_labels = {'Gender': {0: 'Female', 1: 'Male'},
                   'ICUType': {1: 'Coronary', 2: 'Cardiac surgery recovery', 3: 'Medical', 4: 'Surgical'}}
for column, labels in category_labels.items():
    table_data[column] = table_data[column].map(
        lambda value: 'Missing' if pd.isna(value) else labels.get(value, f'Unexpected: {value}'))
groups = {'Overall': table_data,
          'Survived': table_data.loc[table_data[death_column].eq(0)],
          'Died': table_data.loc[table_data[death_column].eq(1)]}
table_columns = {}
for label, group in groups.items():
    entries = {'N': str(len(group))}
    for column in ['Age', 'Height', 'Weight_admission', saps_column, sofa_column]:
        q1, median, q3 = group[column].quantile([0.25, 0.5, 0.75])
        entries[f'{column}, median [Q1, Q3]'] = f'{median:.1f} [{q1:.1f}, {q3:.1f}]'
        entries[f'{column}, missing n'] = str(group[column].isna().sum())
    for column in category_labels:
        for level in sorted(table_data[column].unique()):
            count = int(group[column].eq(level).sum())
            percent = 100 * count / len(group) if len(group) else np.nan
            entries[f'{column}: {level}, n (%)'] = f'{count} ({percent:.1f}%)'
    table_columns[label] = entries
table1 = pd.DataFrame(table_columns)
table1.index.name = 'Characteristic'
display(table1)
table1.to_csv(OUTPUT / 'table1.csv')
del table_data, groups, group
gc.collect()

## Outcome summary

Summarize hospital mortality separately from follow-up Survival. Keep Survival = -1 as a reported no-recorded-death category; summarize durations only for nonnegative entries. This conditional duration summary is not a cohort survival estimate. Length_of_stay covers hospitalization after ICU admission, including time after ICU discharge.

For length of stay and severity scores, report negative codes separately and exclude them only from duration/score summaries. Flag zero/one-day durations for review against the published definitions, retaining recorded nonnegative values. Interpret the resulting medians alongside these code counts and the number of available observations.

In [ ]:
mortality_counts = outcomes[death_column].value_counts().reindex([0, 1], fill_value=0)
outcome_totals = pd.Series({
    'Total N': len(outcomes),
    'Survived hospital discharge': mortality_counts.loc[0],
    'In-hospital deaths': mortality_counts.loc[1],
    'In-hospital mortality (%)': 100 * mortality_counts.loc[1] / len(outcomes),
})
display(outcome_totals)
summary_records = []
for column in [los_column, saps_column, sofa_column, survival_column]:
    recorded = outcomes[column]
    nonnegative = recorded.loc[recorded.ge(0)]
    quartiles = nonnegative.quantile([0.25, 0.5, 0.75])
    summary_records.append({
        'Variable': column, 'N_nonnegative': len(nonnegative),
        'N_minus1': recorded.eq(-1).sum(), 'N_other_negative': recorded.lt(-1).sum(),
        'N_missing': recorded.isna().sum(), 'median': quartiles.loc[0.5],
        'Q1': quartiles.loc[0.25], 'Q3': quartiles.loc[0.75],
        'min': nonnegative.min(), 'max': nonnegative.max(),
    })
outcome_summary = pd.DataFrame(summary_records).set_index('Variable')
display(outcome_summary)
print('Survival = -1 (no recorded death):', outcomes[survival_column].eq(-1).sum())
print('Length_of_stay of zero or one day:', outcomes[los_column].isin([0, 1]).sum())
print('Survival of zero or one day:', outcomes[survival_column].isin([0, 1]).sum())
outcome_summary.to_csv(OUTPUT / 'outcome_summary.csv')
outcome_totals.to_csv(OUTPUT / 'outcome_totals.csv', header=['Value'])

## Wide-table missingness

Define ever measured as a positive non-missing count for each original parameter, including static descriptors. A single measurement is sufficient for patient-level presence even when most hours are empty. Presence does not establish clinical validity, and missing MechVent records are not recoded as absence of ventilation. Use the ranked table and bar chart to identify high and low availability after running this section.

In [ ]:
ever_measured = set_a_wide.set_index('RecordID')[count_columns].gt(0)
ever_measured.columns = parameters
availability = pd.DataFrame({
    'N_ever_measured': ever_measured.sum(),
    'pct_ever_measured': 100 * ever_measured.mean(),
    'pct_never_measured': 100 * (~ever_measured).mean(),
}).sort_values('pct_never_measured', ascending=False)
availability.index.name = 'Parameter'
display(availability.round(2))
availability.to_csv(OUTPUT / 'patient_parameter_availability.csv')
fig, ax = plt.subplots(figsize=(9, 10))
availability['pct_never_measured'].sort_values().plot.barh(ax=ax, color='steelblue')
ax.set(xlabel='Patients never measured (%)', ylabel='Parameter', xlim=(0, 100),
       title='Patient-level missingness during the first 48 hours')
fig.tight_layout()
fig.savefig(OUTPUT / 'wide_missingness.png', dpi=110)
plt.close(fig)
del fig, ax

## Informative missingness

Compare mortality between patients with and without each parameter recorded. Report group sizes and sort by absolute mortality difference (measured minus not measured). An empty group has undefined mortality, represented by NaN rather than zero. Small groups can yield unstable differences, and these comparisons are unadjusted.

This is an association, not evidence that measurement causes mortality. Clinicians may order tests more often for sicker patients, so measurement presence can contain information about patient state and care processes. The plot shows at most 20 defined differences, in percentage points; review denominators before interpreting their ranking.

In [ ]:
mortality = set_a_wide.set_index('RecordID')[death_column].reindex(ever_measured.index)
presence_records = []
for parameter in parameters:
    present = ever_measured[parameter]
    rate_measured = mortality.loc[present].mean()
    rate_not_measured = mortality.loc[~present].mean()
    presence_records.append({
        'Parameter': parameter, 'N_measured': int(present.sum()),
        'N_not_measured': int((~present).sum()),
        'mortality_measured': rate_measured, 'mortality_not_measured': rate_not_measured,
        'mortality_difference': rate_measured - rate_not_measured,
    })
presence_mortality = pd.DataFrame(presence_records).sort_values(
    'mortality_difference', key=lambda values: values.abs(), ascending=False, na_position='last')
display(presence_mortality.round(4))
presence_mortality.to_csv(OUTPUT / 'presence_mortality.csv', index=False)
plot_rows = presence_mortality.dropna(subset=['mortality_difference']).head(20).sort_values('mortality_difference')
if not plot_rows.empty:
    labels = [f'{row.Parameter} ({row.N_measured}/{row.N_not_measured})' for row in plot_rows.itertuples()]
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(labels, 100 * plot_rows['mortality_difference'], color='steelblue')
    ax.axvline(0, color='black', linewidth=1)
    ax.set(xlabel='Mortality difference (measured minus not measured), percentage points',
           ylabel='Parameter (N measured / N not measured)', title='Measurement presence and mortality')
    fig.tight_layout()
    fig.savefig(OUTPUT / 'informative_missingness.png', dpi=110)
    plt.close(fig)
    del fig, ax

## Challenge 2012 assessment criteria

With death as the positive outcome, `TP/(TP+FN)` is sensitivity/recall: the fraction of actual deaths identified. `TP/(TP+FP)` is precision/positive predictive value: the fraction of predicted deaths that are correct. The Challenge criterion `min(TP/(TP+FN), TP/(TP+FP))` rewards maintaining both because the weaker metric determines the score.

The Hosmer-Lemeshow statistic concerns calibration: agreement between predicted risk and observed outcomes across risk groups. The Challenge additionally normalized this statistic by the spread of predicted risks. No predictive model is built and neither metric is calculated here. Source: [Challenge assessment criteria](https://physionet.org/content/challenge-2012/1.0.0/).

## AI-use statement

*Draft for my review:* I used ChatGPT and Codex to understand assignment requirements, help with Git/Pixi/project setup, draft and revise Python code, debug, learn missing-data concepts, suggest visualizations, and explain programming and statistics. My review process included running earlier code incrementally, checking 4,000 patient records, RecordIDs and long-table dimensions, verifying the 0-48-hour conversion, manually inspecting raw patient files and intermediate outputs, and reviewing generated code. I plan to run the rebuilt cells myself and test the final notebook with Pixi; the rebuilt sections have not yet been executed.

Benefits included faster debugging, useful explanations, and validation suggestions. Generated code can run despite incorrect assumptions, and AI may misunderstand clinical variable semantics. I remain responsible for manually validating the code and analysis and for confirming this disclosure matches my actual use of both tools.

## Final validation

Run this only after manually reviewing the preceding cells. Check required files and the saved wide table using only its identifier column, without reloading the full long CSV. This does not establish that every interpretation is correct. After manual review, the separate command `pixi run --locked checkpoint1` can be used to test the complete notebook and regenerate the HTML report.

In [ ]:
required_outputs = ['set-a_long.csv', 'set-a_wide.csv',
                    'hourly_measurement_proportion.csv', 'temporal_missingness.png']
for filename in required_outputs:
    assert (OUTPUT / filename).is_file(), f'Missing output: {filename}'
saved_ids = pd.read_csv(OUTPUT / 'set-a_wide.csv', usecols=['RecordID'], dtype={'RecordID': 'int32'})
assert len(saved_ids) == 4000
assert saved_ids['RecordID'].nunique() == 4000 and saved_ids['RecordID'].is_unique
assert set(saved_ids['RecordID']) == expected_ids
assert len(set_a_wide) == 4000 and set_a_wide['RecordID'].is_unique
saved_hourly = pd.read_csv(OUTPUT / 'hourly_measurement_proportion.csv', index_col=0)
assert saved_hourly.shape == (41, 49)
assert saved_hourly.ge(0).all().all() and saved_hourly.le(1).all().all()
print('Required files exist; wide-table identifiers and hourly proportions passed validation.')
del saved_ids, saved_hourly
gc.collect()
plt.close('all')